# LoRA Finetuning

Finetunes a LoRA adapter on StableDiffusion 1.5's UNet.


Runtime: Colab A100


Dataset: Persian miniature paintings (Wikimedia)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshaSrirangam/stable-diffusion-rebuilt/blob/main/notebooks/train.ipynb)

## Environment/Setup

In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device) # should be cuda

cuda


In [2]:
# clone repo
!git clone https://github.com/HarshaSrirangam/stable-diffusion-rebuilt.git
%cd stable-diffusion-rebuilt
!git pull

fatal: destination path 'stable-diffusion-rebuilt' already exists and is not an empty directory.
/content/stable-diffusion-rebuilt
Already up to date.


In [3]:
# install dependencies (skip torch/numpy to avoid overwriting Colab's preinstalled packages)
%pip install -e . --no-deps -q
%pip install transformers safetensors tqdm accelerate pyyaml datasets -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for stable-diffusion-rebuilt (pyproject.toml) ... done


In [4]:
# mount drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [5]:
# symlink runs/ and data/ (weights, cache, persian)
!ln -sfn /content/drive/MyDrive/sd-rebuilt/data data
!ln -sfn /content/drive/MyDrive/sd-rebuilt/runs runs

!ls runs data

data:
cache  persian	weights

runs:
base			   naruto_r16_selfcross_lora_test
naruto_r16_selfcross_lora  persian_r16_selfcross_lora


## Training

These configs are written to `configs/lora.yaml`. The `runs/<run_name>` for this run comes from this config. If a run already exists, the `name` field in `config/lora.yaml` should be modified.

In [25]:
%%writefile configs/lora.yaml

name: lora_a16
pretrained_path: data/weights/v1-5-pruned-emaonly.safetensors
device: cuda

dataset: persian

r: 32
alpha: 16
targets:
  desc: selfcrossffn
  layers: ["q1", "k1", "v1", "proj_out1", "q2", "k2", "v2", "proj_out2", "linear_geglu1", "linear_geglu2"]

n_epochs: 10
batch_size: 8
lr: 0.0001
seed: 42
log_interval: 10

Overwriting configs/lora.yaml


In [26]:
# training
!python scripts/train_lora.py


>>> Loading UNet and injecting LoRA layers

>>> Preparing dataset

>>> Training
epoch 1/10: 100% 163/163 [00:38<00:00,  4.21it/s]
epoch 2/10: 100% 163/163 [00:37<00:00,  4.38it/s]
epoch 3/10: 100% 163/163 [00:37<00:00,  4.31it/s]
epoch 4/10: 100% 163/163 [00:37<00:00,  4.39it/s]
epoch 5/10: 100% 163/163 [00:38<00:00,  4.21it/s]
epoch 6/10: 100% 163/163 [00:37<00:00,  4.37it/s]
epoch 7/10: 100% 163/163 [00:37<00:00,  4.31it/s]
epoch 8/10: 100% 163/163 [00:38<00:00,  4.29it/s]
epoch 9/10: 100% 163/163 [00:37<00:00,  4.39it/s]
epoch 10/10: 100% 163/163 [00:37<00:00,  4.35it/s]

>>> Done


In [27]:
import yaml
cfg = yaml.safe_load(open("configs/lora.yaml"))
run_name = f"{cfg['dataset']}_r{cfg['r']}_{cfg['targets']['desc']}_{cfg['name']}"
print(run_name) # must match run directory created in previous cell

persian_r32_selfcrossffn_lora_a16


In [28]:
# eval
!python scripts/evaluate_lora.py --run {run_name}


>>> Loading model

>>> Loading LoRA adapter

>>> Generating eval images
Denoising: 100% 50/50 [00:04<00:00, 11.00it/s]
Denoising: 100% 50/50 [00:04<00:00, 11.82it/s]
Denoising: 100% 50/50 [00:04<00:00, 11.93it/s]
Denoising: 100% 50/50 [00:04<00:00, 11.95it/s]
Denoising: 100% 50/50 [00:05<00:00,  9.77it/s]
Denoising: 100% 50/50 [00:04<00:00, 11.94it/s]
Denoising: 100% 50/50 [00:04<00:00, 11.94it/s]
Denoising: 100% 50/50 [00:04<00:00, 11.90it/s]
Denoising: 100% 50/50 [00:04<00:00, 11.90it/s]
Denoising: 100% 50/50 [00:04<00:00, 11.94it/s]

>>> Computing CLIP prompt adherence

>>> Computing CLIP style adherence

>>> Running UNet on eval dataset

>>> Done. Eval outputs written to /content/stable-diffusion-rebuilt/runs/persian_r32_selfcrossffn_lora_a16/eval
